In [1]:
import pandas as pd
import joblib


In [2]:
model = joblib.load(
    "../models/logistic_regression_model.pkl"
)

print("Model loaded successfully.")

Model loaded successfully.


In [3]:
df = pd.read_csv(
    "../data/credit_dataset_cleaned.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (30000, 23)


In [4]:
features = [
    "Age",
    "Employment_Type",
    "Annual_Income",
    "Employment_Duration_Years",
    "Number_of_Dependents",
    "Loan_Purpose",
    "Loan_Amount",
    "Loan_Tenure_Months",
    "Existing_Loans_Count",
    "Total_Outstanding_Debt",
    "Existing_Monthly_EMI",
    "Debt_to_Income_Ratio",
    "Loan_to_Income_Ratio",
    "Credit_Utilization",
    "Previous_Defaults",
    "Missed_Payments",
    "Maximum_Days_Past_Due",
    "Recent_Credit_Enquiries",
    "Credit_History_Length",
    "Number_of_Credit_Accounts",
    "Payment_History",
    "Credit_Score"
]

X = df[features]

In [5]:
probabilities = model.predict_proba(X)

approve_probability = probabilities[:, 1]

print(approve_probability[:10])

[0.40839749 0.67559514 0.62014049 0.63999635 0.90446737 0.06728087
 0.55020744 0.12761699 0.38072078 0.82899423]


In [6]:
reject_probability = probabilities[:, 0]

print(reject_probability[:10])

[0.59160251 0.32440486 0.37985951 0.36000365 0.09553263 0.93271913
 0.44979256 0.87238301 0.61927922 0.17100577]


In [7]:
risk_score = reject_probability * 100

risk_score = risk_score.round(2)

In [11]:
def risk_category(score):
    if score <= 30:
        return "Low Risk"
    elif score <= 60:
        return "Medium Risk"
    else:
        return "High Risk"

In [12]:
risk_categories = [
    risk_category(score)
    for score in risk_score
]

In [13]:
risk_df = pd.DataFrame({
    "Approve_Probability": (
        approve_probability * 100
    ).round(2),

    "Reject_Probability": (
        reject_probability * 100
    ).round(2),

    "Risk_Score": risk_score,

    "Risk_Category": risk_categories
})

risk_df.head(10)

,Approve_Probability,Reject_Probability,Risk_Score,Risk_Category
0,40.84,59.16,59.16,Medium Risk
1,67.56,32.44,32.44,Medium Risk
2,62.01,37.99,37.99,Medium Risk
3,64.00,36.00,36.00,Medium Risk
4,90.45,9.55,9.55,Low Risk
5,6.73,93.27,93.27,High Risk
6,55.02,44.98,44.98,Medium Risk
7,12.76,87.24,87.24,High Risk
8,38.07,61.93,61.93,High Risk
9,82.90,17.10,17.10,Low Risk


In [14]:
def lending_recommendation(category):
    if category == "Low Risk":
        return "Approve"
    elif category == "Medium Risk":
        return "Review"
    else:
        return "Reject"

In [15]:
recommendations = [
    lending_recommendation(category)
    for category in risk_categories
]

In [16]:
risk_df["Lending_Recommendation"] = recommendations

risk_df.head(10)

,Approve_Probability,Reject_Probability,Risk_Score,Risk_Category,Lending_Recommendation
0,40.84,59.16,59.16,Medium Risk,Review
1,67.56,32.44,32.44,Medium Risk,Review
2,62.01,37.99,37.99,Medium Risk,Review
3,64.00,36.00,36.00,Medium Risk,Review
4,90.45,9.55,9.55,Low Risk,Approve
5,6.73,93.27,93.27,High Risk,Reject
6,55.02,44.98,44.98,Medium Risk,Review
7,12.76,87.24,87.24,High Risk,Reject
8,38.07,61.93,61.93,High Risk,Reject
9,82.90,17.10,17.10,Low Risk,Approve


In [17]:
print("Risk Category Distribution:")
print(risk_df["Risk_Category"].value_counts())

Risk Category Distribution:
Risk_Category
High Risk      17491
Low Risk        6487
Medium Risk     6022
Name: count, dtype: int64


In [18]:
print("\nLending Recommendation Distribution:")
print(risk_df["Lending_Recommendation"].value_counts())


Lending Recommendation Distribution:
Lending_Recommendation
Reject     17491
Approve     6487
Review      6022
Name: count, dtype: int64
